# Hungarian Election Prediction 2026 | Andrási Kristóf - media usage


> **Note:** All cell outputs in this notebook have been cleared before publication. The notebook uses Gemius Audience data, which cannot be redistributed due to a confidentiality obligation. The outputs contained outlet identifiers and statistics derived from that data, so they were removed. Re-running the notebook requires obtaining the Gemius source files separately and placing them in `data/Gemius/`.

### Imports

This block loads the packages used in the notebook.


In [ ]:
# this block loads the packages used in the notebook.
import importlib
import subprocess
import sys

# check and install missing packages
def ensure_packages(packages):
    # packages is a list of package names as strings
    for pkg in packages:
        try:
            # try to import the package
            importlib.import_module(pkg)
        except ImportError:
            # if import fails, install with pip
            print(f"{pkg} not found, installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
            print(f"{pkg} installed.")

# first make sure the core packages are available
ensure_packages(["pandas",
                 "numpy",
                 "openpyxl",
                 "matplotlib",
                 "re",
                 "xlrd"])

# now import the packages you actually want to use
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import re
import numpy as np

### Working Directory

This block sets the main project paths so file loading works the same way each time.


In [ ]:
# this block sets the main project paths so file loading works the same way each time.
def find_project_root(start_path=None):
    start_path = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "scripts" / "final_final_scripts").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from the current working directory.")

here = Path.cwd().resolve()
project_root = find_project_root(here)
data_dir = project_root / "data" / "TokaGabor"

print("here:", here)
print("project_root:", project_root)
print("data_dir:", data_dir)


Interpretation. The printed paths should point to this election project. If they do not, the later loading steps can fail.


### Dataframe import

This block loads the prepared tables written by notebook 01.


In [ ]:
# this block loads the prepared tables written by notebook 01.
# recreate output folder path
created_dir = data_dir.parent / "created"

# 1) statistics.csv → final_statistics_oevk
final_statistics_oevk = pd.read_csv(
    created_dir / "statistics.csv"
)

# 2) oevk_results.csv → final_results_oevk_party
final_results_oevk_party = pd.read_csv(
    created_dir / "oevk_results.csv"
)

# 3) polls.csv → polls
polls = pd.read_csv(
    created_dir / "polls.csv"
)

# 4) global_results.csv → mandates_all_combined
mandates_all_combined = pd.read_csv(
    created_dir / "global_results.csv"
)

# optional: clean up helper path variable
for var in ["created_dir"]:
    if var in locals():
        del locals()[var]
del var

Summary table for media availability across years aggregated by window (w1, w2, w3) and metric (real users, views, time) because showing all would be too long and there is no NA across the same media variable.


In [ ]:
# summary table for media availability across years
# aggregated by window (w1, w2, w3) and metric (real users, views, time)
# because showing all would be too long and there is no NA across the same media variable

media_cols = [c for c in final_statistics_oevk.columns if any(x in c for x in ["real_users_", "views_", "time_per_view_s_"])]
outlets_map = {}
for c in media_cols:
    original_outlet = re.sub(r"^(?:real_users|views|time_per_view_s)_w[1-3]_", "", c)
    # Aggregate and clean names for display
    cleaned_name = original_outlet.replace("_b", "").replace("_com", "").replace("_hu", "").replace("_app", "").capitalize()
    if cleaned_name not in outlets_map:
        outlets_map[cleaned_name] = set()
    outlets_map[cleaned_name].add(original_outlet)

results = []
for cleaned_name in sorted(outlets_map.keys()):
    row = {"Media outlet": cleaned_name}
    for year in [2022, 2024, 2026]:
        df_year = final_statistics_oevk[final_statistics_oevk["year"] == year]

        # Calculate non-missing percentage
        # We take the minimum percentage among all related variables for this outlet
        all_related_cols = []
        for original_outlet in outlets_map[cleaned_name]:
            related_cols = [c for c in media_cols if c.endswith("_" + original_outlet)]
            all_related_cols.extend(related_cols)

        if all_related_cols:
            # Calculate mean of non-missing for all these columns and take the minimum
            min_pct = df_year[all_related_cols].notna().mean().min() * 100
        else:
            min_pct = 0.0

        row[str(year)] = f"{min_pct:.1f}%"
    results.append(row)

df_media_availability = pd.DataFrame(results)

# display
display(df_media_availability)

# Export to LaTeX
try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

project_root = here
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

latex_output_paths = [
    project_root / "graphs_tables" / "media_availability_summary.tex",
    project_root / "doc" / "ÚJLaTex_ENGLISH_Template másolat 2" / "figures" / "media_availability_summary.tex",
]
for latex_path in latex_output_paths:
    latex_path.parent.mkdir(parents=True, exist_ok=True)

latex_string = df_media_availability.to_latex(
    index=False,
    caption="Media availability minimum non-missing percentage by year (aggregated)",
    label="tab:media_availability_summary",
)
for latex_path in latex_output_paths:
    latex_path.write_text(latex_string, encoding="utf-8")

print("Saved LaTeX table to:")
for latex_path in latex_output_paths:
    print(latex_path)

# Clean up
for var in ["media_cols", "outlets_map", "results", "cleaned_name", "row", "year", "df_year", "min_pct", "original_outlet", "all_related_cols", "latex_output_paths", "latex_path", "latex_string", "project_root", "here", "df_media_availability"]:
    if var in locals():
        del locals()[var]
del var


Interpretation. This output is a quick diagnostic check of scale, spread and completeness.


This block prepares the next part of the notebook.


In [ ]:
# this block prepares the next part of the notebook.


### Media variable screening for later modelling

This block is a first screen for the media variables.

The goal is not to choose a final model here. The goal is to see:

- which years have usable media data
- whether `w1`, `w2`, or `w3` looks stronger
- which example outlet variables have a stable relationship with election outcomes
- and whether the windows are so similar that using all of them would likely overfit


In [ ]:
# this block screens media variables to see which ones are usable later.
block_order = ["gov", "opp", "opp_radical", "other"]
selected_outlets = [
    "mediaklikk_b",
    "hirado_b",
    "tv2_b",
    "tenyek_b",
    "magyarnemzet_b",
    "mandiner_b",
    "origo_b",
    "telex_b",
    "444_b",
    "hvg_b",
    "rtl_b",
    "24_b",
]
metric_specs = {
    "real_users": "per_1000_log",
    "views": "per_1000_log",
    "time_per_view_s": "log1p",
}
windows = ["w1", "w2", "w3"]


def transform_media(series, pop_total, rule):
    series = pd.to_numeric(series, errors="coerce")
    pop_total = pd.to_numeric(pop_total, errors="coerce")
    if rule == "per_1000_log":
        return np.log1p(series.mul(1000.0).div(pop_total).replace([np.inf, -np.inf], np.nan))
    return np.log1p(series)


screen_results = final_results_oevk_party.copy()
screen_results["list_share"] = pd.to_numeric(screen_results["list_share"], errors="coerce")

block_panel = (
    screen_results
    .groupby(["year", "election_type", "oevk_id", "party_block"], as_index=False)["list_share"]
    .sum()
    .pivot(index=["year", "election_type", "oevk_id"], columns="party_block", values="list_share")
    .reset_index()
)

for block in block_order:
    if block not in block_panel.columns:
        block_panel[block] = 0.0

block_panel["gov_margin"] = (
    block_panel["gov"].fillna(0)
    - block_panel[["opp", "opp_radical", "other"]].fillna(0).sum(axis=1)
)

media_panel = final_statistics_oevk.merge(
    block_panel[["year", "election_type", "oevk_id", "gov", "opp", "opp_radical", "other", "gov_margin"]],
    on=["year", "oevk_id"],
    how="inner",
)

media_panel = media_panel[media_panel["year"].isin([2022, 2024])].copy()
media_panel["pop_total"] = pd.to_numeric(media_panel["pop_total"], errors="coerce").replace({0: np.nan})

availability_rows = []
for year, sub in final_statistics_oevk.groupby("year"):
    for window in windows:
        selected_cols = []
        for metric in metric_specs:
            for outlet in selected_outlets:
                col = f"{metric}_{window}_{outlet}"
                if col in sub.columns:
                    selected_cols.append(col)

        if not selected_cols:
            continue

        availability_rows.append(
            {
                "year": int(year),
                "window": window,
                "selected_media_vars": len(selected_cols),
                "avg_nonmissing_share": sub[selected_cols].notna().mean().mean(),
                "rows_with_any_selected_media": int(sub[selected_cols].notna().any(axis=1).sum()),
            }
        )

availability = pd.DataFrame(availability_rows)

corr_rows = []
for year in [2022, 2024]:
    sub = media_panel[media_panel["year"] == year].copy()

    for metric, rule in metric_specs.items():
        for window in windows:
            for outlet in selected_outlets:
                col = f"{metric}_{window}_{outlet}"
                if col not in sub.columns:
                    continue

                x = transform_media(sub[col], sub["pop_total"], rule)
                if x.notna().sum() < 30:
                    continue
                if x.dropna().std() == 0:
                    continue

                for target in ["gov", "opp", "opp_radical", "gov_margin"]:
                    valid = x.notna() & sub[target].notna()
                    if valid.sum() < 30:
                        continue

                    corr = x[valid].corr(sub.loc[valid, target])
                    corr_rows.append(
                        {
                            "year": year,
                            "metric": metric,
                            "window": window,
                            "outlet": outlet,
                            "target": target,
                            "n": int(valid.sum()),
                            "corr": float(corr),
                            "abs_corr": float(abs(corr)),
                        }
                    )

media_corr = pd.DataFrame(corr_rows)

window_signal = (
    media_corr[media_corr["target"] == "gov_margin"]
    .groupby(["metric", "window"], as_index=False)
    .agg(
        mean_abs_corr=("abs_corr", "mean"),
        median_abs_corr=("abs_corr", "median"),
        max_abs_corr=("abs_corr", "max"),
    )
)

stable_rows = []
for (metric, window, outlet, target), grp in media_corr.groupby(["metric", "window", "outlet", "target"]):
    grp = grp.sort_values("year")
    if grp["year"].nunique() != 2:
        continue
    if grp["corr"].isna().any():
        continue

    stable_rows.append(
        {
            "metric": metric,
            "window": window,
            "outlet": outlet,
            "target": target,
            "corr_2022": float(grp.iloc[0]["corr"]),
            "corr_2024": float(grp.iloc[1]["corr"]),
            "same_sign": bool(np.sign(grp.iloc[0]["corr"]) == np.sign(grp.iloc[1]["corr"])),
            "mean_abs_corr": float(grp["abs_corr"].mean()),
        }
    )

stable_corr = pd.DataFrame(stable_rows)

top_media_candidates = (
    stable_corr[(stable_corr["target"] == "gov_margin") & (stable_corr["same_sign"])]
    .sort_values(["mean_abs_corr", "window", "metric"], ascending=[False, True, True])
    .head(12)
    .copy()
)

top_media_candidates["feature"] = (
    top_media_candidates["metric"]
    + " "
    + top_media_candidates["window"]
    + " "
    + top_media_candidates["outlet"]
)

top_media_by_target = (
    stable_corr[stable_corr["same_sign"]]
    .sort_values(["target", "mean_abs_corr"], ascending=[True, False])
    .groupby("target", as_index=False, group_keys=False)
    .head(4)
    .copy()
)

top_media_by_target["feature"] = (
    top_media_by_target["metric"]
    + " "
    + top_media_by_target["window"]
    + " "
    + top_media_by_target["outlet"]
)

redundancy_rows = []
for metric, rule in metric_specs.items():
    for outlet in selected_outlets:
        transformed = {}
        for window in windows:
            col = f"{metric}_{window}_{outlet}"
            if col not in media_panel.columns:
                continue
            transformed[window] = transform_media(media_panel[col], media_panel["pop_total"], rule)

        for left, right in [("w1", "w2"), ("w1", "w3"), ("w2", "w3")]:
            if left not in transformed or right not in transformed:
                continue

            valid = transformed[left].notna() & transformed[right].notna()
            if valid.sum() < 30:
                continue

            redundancy_rows.append(
                {
                    "metric": metric,
                    "pair": f"{left} vs {right}",
                    "outlet": outlet,
                    "corr": float(transformed[left][valid].corr(transformed[right][valid])),
                }
            )

media_redundancy = pd.DataFrame(redundancy_rows)
redundancy_summary = (
    media_redundancy
    .groupby(["metric", "pair"], as_index=False)
    .agg(
        median_corr=("corr", "median"),
        mean_corr=("corr", "mean"),
        min_corr=("corr", "min"),
        max_corr=("corr", "max"),
    )
)

model_size_check = pd.DataFrame(
    [
        {
            "setup": "usable district-election rows with media and outcomes",
            "count": int(len(media_panel)),
        },
        {
            "setup": "all raw media variables in statistics table",
            "count": int(
                sum(
                    c.startswith(("real_users_", "views_", "time_per_view_s_"))
                    for c in final_statistics_oevk.columns
                )
            ),
        },
        {
            "setup": "selected example outlets times all metrics times all windows",
            "count": int(
                sum(
                    1
                    for metric in metric_specs
                    for window in windows
                    for outlet in selected_outlets
                    if f"{metric}_{window}_{outlet}" in final_statistics_oevk.columns
                )
            ),
        },
        {
            "setup": "selected example outlets with one metric and one window",
            "count": int(len(selected_outlets)),
        },
    ]
)

display(availability.round(3))
display(model_size_check)
display(window_signal.round(3))
display(top_media_candidates[["feature", "corr_2022", "corr_2024", "mean_abs_corr"]].round(3))
display(top_media_by_target[["target", "feature", "corr_2022", "corr_2024", "mean_abs_corr"]].round(3))
display(redundancy_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for metric in window_signal["metric"].unique():
    sub = window_signal[window_signal["metric"] == metric]
    axes[0].plot(sub["window"], sub["mean_abs_corr"], marker="o", linewidth=2, label=metric)

axes[0].set_title("mean absolute correlation by media window")
axes[0].set_ylabel("mean absolute correlation with gov margin")
axes[0].set_xlabel("media window")
axes[0].grid(axis="y")
axes[0].legend(frameon=False)

plot_top = top_media_candidates.sort_values("mean_abs_corr", ascending=True)
axes[1].barh(plot_top["feature"], plot_top["mean_abs_corr"], color="#5b8a72")
axes[1].set_title("top stable example media variables")
axes[1].set_xlabel("mean absolute correlation with gov margin")
axes[1].grid(axis="x")

plt.tight_layout()
plt.show()
# clean up temporary variables, keep the reusable media screening objects
for var in [
    "block",
    "block_order",
    "block_panel",
    "availability_rows",
    "col",
    "corr",
    "corr_rows",
    "fig",
    "axes",
    "grp",
    "left",
    "media_redundancy",
    "outlet",
    "plot_top",
    "redundancy_rows",
    "right",
    "rule",
    "screen_results",
    "selected_cols",
    "stable_rows",
    "sub",
    "target",
    "transformed",
    "valid",
    "window",
    "x",
    "year",
]:
    if var in locals():
        del locals()[var]
del var


Interpretation. Lower coverage means the variable is harder to use safely without imputation.


This block makes a figure to show the main pattern in the data.


In [ ]:
# this block makes a figure to show the main pattern in the data.
# for exporting the left graph

# set global font to Times New Roman, size 12
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 12

block_order = ["gov", "opp", "opp_radical", "other"]
selected_outlets = [
    "mediaklikk_b",
    "hirado_b",
    "tv2_b",
    "tenyek_b",
    "magyarnemzet_b",
    "mandiner_b",
    "origo_b",
    "telex_b",
    "444_b",
    "hvg_b",
    "rtl_b",
    "24_b",
]
metric_specs = {
    "real_users": "per_1000_log",
    "views": "per_1000_log",
    "time_per_view_s": "log1p",
}
metric_labels = {
    "real_users": "Real users",
    "views": "Views",
    "time_per_view_s": "Time per view (s)",
}
windows = ["w1", "w2", "w3"]


def transform_media(series, pop_total, rule):
    series = pd.to_numeric(series, errors="coerce")
    pop_total = pd.to_numeric(pop_total, errors="coerce")
    if rule == "per_1000_log":
        return np.log1p(series.mul(1000.0).div(pop_total).replace([np.inf, -np.inf], np.nan))
    return np.log1p(series)


screen_results = final_results_oevk_party.copy()
screen_results["list_share"] = pd.to_numeric(screen_results["list_share"], errors="coerce")

block_panel = (
    screen_results
    .groupby(["year", "election_type", "oevk_id", "party_block"], as_index=False)["list_share"]
    .sum()
    .pivot(index=["year", "election_type", "oevk_id"], columns="party_block", values="list_share")
    .reset_index()
)

for block in block_order:
    if block not in block_panel.columns:
        block_panel[block] = 0.0

block_panel["gov_margin"] = (
    block_panel["gov"].fillna(0)
    - block_panel[["opp", "opp_radical", "other"]].fillna(0).sum(axis=1)
)

media_panel = final_statistics_oevk.merge(
    block_panel[["year", "election_type", "oevk_id", "gov", "opp", "opp_radical", "other", "gov_margin"]],
    on=["year", "oevk_id"],
    how="inner",
)

media_panel = media_panel[media_panel["year"].isin([2022, 2024])].copy()
media_panel["pop_total"] = pd.to_numeric(media_panel["pop_total"], errors="coerce").replace({0: np.nan})

corr_rows = []
for year in [2022, 2024]:
    sub = media_panel[media_panel["year"] == year].copy()

    for metric, rule in metric_specs.items():
        for window in windows:
            for outlet in selected_outlets:
                col = f"{metric}_{window}_{outlet}"
                if col not in sub.columns:
                    continue

                x = transform_media(sub[col], sub["pop_total"], rule)
                if x.notna().sum() < 30:
                    continue
                if x.dropna().std() == 0:
                    continue

                valid = x.notna() & sub["gov_margin"].notna()
                if valid.sum() < 30:
                    continue

                corr = x[valid].corr(sub.loc[valid, "gov_margin"])
                corr_rows.append(
                    {
                        "year": year,
                        "metric": metric,
                        "window": window,
                        "outlet": outlet,
                        "n": int(valid.sum()),
                        "corr": float(corr),
                        "abs_corr": float(abs(corr)),
                    }
                )

media_corr_plot = pd.DataFrame(corr_rows)

window_signal_plot = (
    media_corr_plot
    .groupby(["metric", "window"], as_index=False)
    .agg(
        mean_abs_corr=("abs_corr", "mean"),
        median_abs_corr=("abs_corr", "median"),
        max_abs_corr=("abs_corr", "max"),
    )
)

# create only the left-side plot
fig, ax = plt.subplots(figsize=(6.5, 4.5))

colors_map = {
    "real_users": "#edc948",
    "views": "#e15759",
    "time_per_view_s": "#9c9c9c",
}

for metric in window_signal_plot["metric"].unique():
    sub = window_signal_plot[window_signal_plot["metric"] == metric]
    ax.plot(
        sub["window"],
        sub["mean_abs_corr"],
        marker="o",
        linewidth=2,
        label=metric_labels.get(metric, metric),
        color=colors_map.get(metric, None),
    )

ax.set_ylabel("Mean absolute correlation with gov margin", fontsize=12)
ax.set_xlabel("Media window (w3: last 2 months, w2: 3-6 months before election, w1: 7-12 months)", fontsize=12)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=12)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()

# export the figure
if "project_root" not in locals():
    try:
        here = Path(__file__).resolve().parent
    except NameError:
        here = Path.cwd()

    project_root = here
    while not (project_root / "data").exists() and project_root != project_root.parent:
        project_root = project_root.parent

output_fig_paths = [
    project_root / "graphs_tables" / "media_window_signal.png",
    project_root / "doc" / "ÚJLaTex_ENGLISH_Template másolat 2" / "figures" / "media_window_signal.png",
]
for output_fig_path in output_fig_paths:
    output_fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure to:")
for output_fig_path in output_fig_paths:
    print(output_fig_path)


### Easy English interpretation of the media screening block

This block is only a first screen. It does not prove causality, and it does not choose the final model by itself.

What to look for:

- The availability table tells us that real media screening only starts from `2022` and `2024`. For `2018` and `2019`, the media block is basically missing.
- The size table shows the overfitting risk. We only have a small number of district-election rows with usable media and outcomes, but the raw media table has many possible variables.
- The window table tells us whether `w1`, `w2`, or `w3` looks stronger on average. If the values are very close, then there is no strong reason to keep all windows.
- The top-feature table shows example outlet variables that are relatively stable across `2022` and `2024`.
- The redundancy table is very important. If `w1`, `w2`, and `w3` are highly correlated with each other for the same outlet and metric, then using all three together can easily overfit.

So the practical lesson is usually this:

- do not throw all media variables into the model
- prefer a small set of example outlets
- often choose one window per metric or one window per outlet
- and keep regularization if several related media variables stay in the model


### Media variable screening with only `w1` variables

This repeats the same idea as the previous block, but now I only look at `w1` variables.

The point is simple:

- if I want a smaller media block
- and I want to inspect the `w1` window on its own
- then I should check the `w1` variables on their own


In [ ]:
# this block screens media variables to see which ones are usable later.
w1_window = "w1"

availability_w1_rows = []
selected_w1_cols = []
for metric in metric_specs:
    for outlet in selected_outlets:
        col = f"{metric}_{w1_window}_{outlet}"
        if col in final_statistics_oevk.columns:
            selected_w1_cols.append(col)

for year, sub in final_statistics_oevk.groupby("year"):
    if not selected_w1_cols:
        continue

    availability_w1_rows.append(
        {
            "year": int(year),
            "selected_media_vars": len(selected_w1_cols),
            "avg_nonmissing_share": sub[selected_w1_cols].notna().mean().mean(),
            "rows_with_any_selected_media": int(sub[selected_w1_cols].notna().any(axis=1).sum()),
        }
    )

availability_w1 = pd.DataFrame(availability_w1_rows)

if "stable_corr" not in locals() or "media_corr" not in locals() or "target" not in media_corr.columns:
    corr_rows = []
    for year in [2022, 2024]:
        sub = media_panel[media_panel["year"] == year].copy()

        for metric, rule in metric_specs.items():
            for window in windows:
                for outlet in selected_outlets:
                    col = f"{metric}_{window}_{outlet}"
                    if col not in sub.columns:
                        continue

                    x = transform_media(sub[col], sub["pop_total"], rule)
                    if x.notna().sum() < 30:
                        continue
                    if x.dropna().std() == 0:
                        continue

                    for target in ["gov", "opp", "opp_radical", "gov_margin"]:
                        valid = x.notna() & sub[target].notna()
                        if valid.sum() < 30:
                            continue

                        corr = x[valid].corr(sub.loc[valid, target])
                        corr_rows.append(
                            {
                                "year": year,
                                "metric": metric,
                                "window": window,
                                "outlet": outlet,
                                "target": target,
                                "n": int(valid.sum()),
                                "corr": float(corr),
                                "abs_corr": float(abs(corr)),
                            }
                        )

    media_corr = pd.DataFrame(corr_rows)

    stable_rows = []
    for (metric, window, outlet, target), grp in media_corr.groupby(["metric", "window", "outlet", "target"]):
        grp = grp.sort_values("year")
        if grp["year"].nunique() != 2:
            continue
        if grp["corr"].isna().any():
            continue

        stable_rows.append(
            {
                "metric": metric,
                "window": window,
                "outlet": outlet,
                "target": target,
                "corr_2022": float(grp.iloc[0]["corr"]),
                "corr_2024": float(grp.iloc[1]["corr"]),
                "same_sign": bool(np.sign(grp.iloc[0]["corr"]) == np.sign(grp.iloc[1]["corr"])),
                "mean_abs_corr": float(grp["abs_corr"].mean()),
            }
        )

    stable_corr = pd.DataFrame(stable_rows)

w1_corr = media_corr[media_corr["window"] == w1_window].copy()

metric_signal_w1 = (
    w1_corr[w1_corr["target"] == "gov_margin"]
    .groupby("metric", as_index=False)
    .agg(
        mean_abs_corr=("abs_corr", "mean"),
        median_abs_corr=("abs_corr", "median"),
        max_abs_corr=("abs_corr", "max"),
    )
    .sort_values("mean_abs_corr", ascending=False)
)

top_w1_candidates = (
    stable_corr[
        (stable_corr["window"] == w1_window)
        & (stable_corr["target"] == "gov_margin")
        & (stable_corr["same_sign"])
    ]
    .sort_values(["mean_abs_corr", "metric", "outlet"], ascending=[False, True, True])
    .head(12)
    .copy()
)

top_w1_candidates["feature"] = (
    top_w1_candidates["metric"]
    + " "
    + top_w1_candidates["outlet"]
)

top_w1_by_target = (
    stable_corr[
        (stable_corr["window"] == w1_window)
        & (stable_corr["same_sign"])
    ]
    .sort_values(["target", "mean_abs_corr"], ascending=[True, False])
    .groupby("target", as_index=False, group_keys=False)
    .head(4)
    .copy()
)

top_w1_by_target["feature"] = (
    top_w1_by_target["metric"]
    + " "
    + top_w1_by_target["outlet"]
)

w1_feature_frame = pd.DataFrame(index=media_panel.index)
for metric, rule in metric_specs.items():
    for outlet in selected_outlets:
        col = f"{metric}_{w1_window}_{outlet}"
        if col not in media_panel.columns:
            continue
        feature_name = f"{metric} {outlet}"
        w1_feature_frame[feature_name] = transform_media(media_panel[col], media_panel["pop_total"], rule)

pair_rows = []
feature_cols = w1_feature_frame.columns.tolist()
for i, left in enumerate(feature_cols):
    for right in feature_cols[i + 1:]:
        valid = w1_feature_frame[left].notna() & w1_feature_frame[right].notna()
        if valid.sum() < 30:
            continue
        corr = w1_feature_frame.loc[valid, left].corr(w1_feature_frame.loc[valid, right])
        pair_rows.append(
            {
                "left_feature": left,
                "right_feature": right,
                "corr": float(corr),
                "abs_corr": float(abs(corr)),
            }
        )

w1_pair_corr = pd.DataFrame(
    pair_rows,
    columns=["left_feature", "right_feature", "corr", "abs_corr"],
)
w1_pair_summary = pd.DataFrame(
    {
        "summary": [
            "median absolute pair correlation among w1 candidates",
            "mean absolute pair correlation among w1 candidates",
            "max absolute pair correlation among w1 candidates",
        ],
        "value": [
            w1_pair_corr["abs_corr"].median(),
            w1_pair_corr["abs_corr"].mean(),
            w1_pair_corr["abs_corr"].max(),
        ],
    }
)

top_w1_pair_corr = w1_pair_corr.sort_values("abs_corr", ascending=False).head(12).copy()

model_size_check_w1 = pd.DataFrame(
    [
        {
            "setup": "usable district-election rows with media and outcomes",
            "count": int(len(media_panel)),
        },
        {
            "setup": "selected w1 example variables",
            "count": int(len(selected_w1_cols)),
        },
        {
            "setup": "top stable w1 candidates shown below",
            "count": int(len(top_w1_candidates)),
        },
    ]
)

display(availability_w1.round(3))
display(model_size_check_w1)
display(metric_signal_w1.round(3))
display(top_w1_candidates[["feature", "corr_2022", "corr_2024", "mean_abs_corr"]].round(3))
display(top_w1_by_target[["target", "feature", "corr_2022", "corr_2024", "mean_abs_corr"]].round(3))
display(w1_pair_summary.round(3))
display(top_w1_pair_corr[["left_feature", "right_feature", "corr", "abs_corr"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].bar(metric_signal_w1["metric"], metric_signal_w1["mean_abs_corr"], color=["#5b8a72", "#4e79a7", "#d1495b"])
axes[0].set_title("w1 mean absolute correlation by metric")
axes[0].set_ylabel("mean absolute correlation with gov margin")
axes[0].set_xlabel("metric")
axes[0].grid(axis="y")

plot_top_w1 = top_w1_candidates.sort_values("mean_abs_corr", ascending=True)
axes[1].barh(plot_top_w1["feature"], plot_top_w1["mean_abs_corr"], color="#5b8a72")
axes[1].set_title("top stable w1 media variables")
axes[1].set_xlabel("mean absolute correlation with gov margin")
axes[1].grid(axis="x")

plt.tight_layout()
plt.show()
# clean up temporary variables, keep only the w1 summary objects
for var in [
    "availability_w1_rows",
    "axes",
    "col",
    "corr",
    "feature_cols",
    "feature_name",
    "fig",
    "i",
    "left",
    "metric",
    "outlet",
    "pair_rows",
    "plot_top_w1",
    "right",
    "selected_w1_cols",
    "sub",
    "valid",
    "w1_feature_frame",
    "w1_window",
    "year",
]:
    if var in locals():
        del locals()[var]
del var


### Easy English interpretation of the `w1`-only media screening block

This block asks a narrower question.

If I only want to inspect the `w1` media window on its own, which `w1` variables look most useful?

What to look for:

- The availability table again tells us that this analysis is really based on `2022` and `2024`, not on the older elections.
- The metric table tells us whether `real_users`, `views`, or `time_per_view_s` looks strongest when I only keep `w1` variables.
- The top-feature tables show the `w1` outlet variables that look relatively stable across the two usable election years.
- The pair-correlation tables matter for overfitting. Even if I only keep `w1`, many candidate variables can still be strongly correlated with each other.

So the practical lesson here is:

- a `w1`-only media block is simpler and easier to defend
- but I still should not throw in every `w1` variable
- it is better to choose a small number of stable example outlets
- and keep regularization in the later predictive model


### Step by step shortlist with only `w1 real_users` variables

This block does the selection in the exact order below.

1. Rank the `w1 real_users` variables by `mean_abs_corr`.
2. Keep only the variables that point in the same direction in both usable election years.
3. Drop variables that are too similar to already selected ones.

So yes, `mean_abs_corr` is still useful here.

It is the first ranking rule, but it is not the final rule.
A variable can have a high `mean_abs_corr` and still be a bad final choice if it changes sign or if it is almost the same as another selected variable.


In [ ]:
# this block builds or checks a smaller shortlist of variables for later modelling.
selection_threshold = 0.90

real_users_w1_ranked = (
    stable_corr[
        (stable_corr["metric"] == "real_users")
        & (stable_corr["window"] == "w1")
        & (stable_corr["target"] == "gov_margin")
    ]
    .sort_values("mean_abs_corr", ascending=False)
    .reset_index(drop=True)
    .copy()
)

real_users_w1_ranked["rank"] = np.arange(1, len(real_users_w1_ranked) + 1)
real_users_w1_ranked = real_users_w1_ranked[
    ["rank", "outlet", "corr_2022", "corr_2024", "same_sign", "mean_abs_corr"]
]

real_users_w1_same_sign = real_users_w1_ranked[real_users_w1_ranked["same_sign"]].reset_index(drop=True).copy()

real_users_w1_features = {}
for outlet in selected_outlets:
    col = f"real_users_w1_{outlet}"
    if col in media_panel.columns:
        real_users_w1_features[outlet] = transform_media(media_panel[col], media_panel["pop_total"], "per_1000_log")

selected_real_users_w1 = []
selection_rows = []

for _, row in real_users_w1_same_sign.iterrows():
    outlet = row["outlet"]
    keep = True
    closest_selected = pd.NA
    abs_pair_corr = pd.NA

    for chosen in selected_real_users_w1:
        valid = real_users_w1_features[outlet].notna() & real_users_w1_features[chosen].notna()
        if valid.sum() < 30:
            continue

        pair_corr = abs(real_users_w1_features[outlet][valid].corr(real_users_w1_features[chosen][valid]))

        if pd.isna(abs_pair_corr) or pair_corr > abs_pair_corr:
            abs_pair_corr = float(pair_corr)
            closest_selected = chosen

        if pair_corr >= selection_threshold:
            keep = False
            break

    if keep:
        selected_real_users_w1.append(outlet)

    selection_rows.append(
        {
            "rank": int(row["rank"]),
            "outlet": outlet,
            "mean_abs_corr": float(row["mean_abs_corr"]),
            "corr_2022": float(row["corr_2022"]),
            "corr_2024": float(row["corr_2024"]),
            "decision": "keep" if keep else "drop as too similar",
            "closest_selected": closest_selected,
            "abs_pair_corr_to_selected": abs_pair_corr,
        }
    )

real_users_w1_selection_trace = pd.DataFrame(selection_rows)

real_users_w1_shortlist = pd.DataFrame(
    {
        "selected_outlet": selected_real_users_w1,
        "selection_threshold": [selection_threshold] * len(selected_real_users_w1),
    }
)

shortlist_pair_rows = []
for i, left in enumerate(selected_real_users_w1):
    for right in selected_real_users_w1[i + 1:]:
        valid = real_users_w1_features[left].notna() & real_users_w1_features[right].notna()
        if valid.sum() < 30:
            continue
        shortlist_pair_rows.append(
            {
                "left_outlet": left,
                "right_outlet": right,
                "pair_corr": float(real_users_w1_features[left][valid].corr(real_users_w1_features[right][valid])),
            }
        )

real_users_w1_shortlist_pairs = pd.DataFrame(shortlist_pair_rows)

display(real_users_w1_ranked.round(3))
display(real_users_w1_same_sign.round(3))
display(real_users_w1_selection_trace.round(3))
display(real_users_w1_shortlist)
display(real_users_w1_shortlist_pairs.round(3))

fig, ax = plt.subplots(figsize=(9, 4.8))
plot_rank = real_users_w1_ranked.sort_values("mean_abs_corr", ascending=True)
colors = ["#5b8a72" if outlet in selected_real_users_w1 else "#c8c8c8" for outlet in plot_rank["outlet"]]
ax.barh(plot_rank["outlet"], plot_rank["mean_abs_corr"], color=colors)
ax.set_title("w1 real_users ranking and final shortlist")
ax.set_xlabel("mean absolute correlation with gov margin")
ax.grid(axis="x")
plt.tight_layout()
plt.show()
# clean up temporary variables, keep only the real_users w1 shortlist objects
for var in [
    "abs_pair_corr",
    "ax",
    "chosen",
    "closest_selected",
    "col",
    "colors",
    "fig",
    "i",
    "keep",
    "left",
    "outlet",
    "pair_corr",
    "plot_rank",
    "right",
    "real_users_w1_features",
    "row",
    "selection_rows",
    "selection_threshold",
    "shortlist_pair_rows",
    "valid",
]:
    if var in locals():
        del locals()[var]
del var


### Easy English interpretation of the `w1 real_users` shortlist block

This is the strictest version of the screening.

How to read it:

- The first table is only a ranking. A higher `mean_abs_corr` is better, but this alone does not decide the final shortlist.
- The second table keeps only variables that point in the same direction in both usable election years.
- The third table shows the greedy selection step. A variable is dropped if it is too similar to one that was already kept.
- The shortlist table is the final result of this rule.

So yes, `mean_abs_corr` still matters.

It is useful as the first ranking score.
But it should not be the only rule.
The final choice should also check stability across years and redundancy with the variables already selected.


### Step by step shortlist with all `w1` media variables

This is the same check again, but now I use every `w1` variable from three groups:

- `real_users_w1_*`
- `views_w1_*`
- `time_per_view_s_w1_*`

The order stays the same.

1. Rank by `mean_abs_corr`.
2. Keep only variables with the same sign in both usable election years.
3. Drop variables that are too similar to variables already selected.

So this block answers a broader question than the previous one.

If I let all `w1` media types compete with each other, which variables survive the same screening logic?


In [ ]:
# this block builds or checks a smaller shortlist of variables for later modelling.
all_w1_selection_threshold = 0.90
all_w1_prefix_specs = {
    "real_users": "real_users_w1_",
    "views": "views_w1_",
    "time_per_view_s": "time_per_view_s_w1_",
}
all_w1_transform_rule = {
    "real_users": "per_1000_log",
    "views": "per_1000_log",
    "time_per_view_s": "log1p",
}

all_w1_rows = []
all_w1_features = {}

for metric, prefix in all_w1_prefix_specs.items():
    metric_cols = [c for c in media_panel.columns if c.startswith(prefix)]

    for col in metric_cols:
        outlet = col[len(prefix):]
        transformed_full = transform_media(media_panel[col], media_panel["pop_total"], all_w1_transform_rule[metric])
        all_w1_features[f"{metric} {outlet}"] = transformed_full

        year_corr = {}
        ok = True
        same_sign = True
        sign_ref = None
        abs_corrs = []

        for year in [2022, 2024]:
            sub = media_panel[media_panel["year"] == year].copy()
            transformed_year = transform_media(sub[col], sub["pop_total"], all_w1_transform_rule[metric])
            valid = transformed_year.notna() & sub["gov_margin"].notna()

            if valid.sum() < 30:
                ok = False
                break
            if transformed_year[valid].std() == 0:
                ok = False
                break

            corr = float(transformed_year[valid].corr(sub.loc[valid, "gov_margin"]))
            year_corr[year] = corr
            abs_corrs.append(abs(corr))

            sign_now = np.sign(corr)
            if sign_ref is None:
                sign_ref = sign_now
            elif sign_now != sign_ref:
                same_sign = False

        if not ok:
            continue

        all_w1_rows.append(
            {
                "metric": metric,
                "outlet": outlet,
                "feature": f"{metric} {outlet}",
                "corr_2022": year_corr[2022],
                "corr_2024": year_corr[2024],
                "same_sign": same_sign,
                "mean_abs_corr": float(np.mean(abs_corrs)),
            }
        )

all_w1_ranked = pd.DataFrame(all_w1_rows).sort_values("mean_abs_corr", ascending=False).reset_index(drop=True)
all_w1_ranked["rank"] = np.arange(1, len(all_w1_ranked) + 1)
all_w1_ranked = all_w1_ranked[
    ["rank", "metric", "outlet", "feature", "corr_2022", "corr_2024", "same_sign", "mean_abs_corr"]
]

all_w1_same_sign = all_w1_ranked[all_w1_ranked["same_sign"]].reset_index(drop=True).copy()

all_w1_selected = []
all_w1_trace_rows = []

for _, row in all_w1_same_sign.iterrows():
    feature = row["feature"]
    keep = True
    closest_selected = pd.NA
    abs_pair_corr = pd.NA

    for chosen in all_w1_selected:
        valid = all_w1_features[feature].notna() & all_w1_features[chosen].notna()
        if valid.sum() < 30:
            continue

        pair_corr = abs(float(all_w1_features[feature][valid].corr(all_w1_features[chosen][valid])))

        if pd.isna(abs_pair_corr) or pair_corr > abs_pair_corr:
            abs_pair_corr = pair_corr
            closest_selected = chosen

        if pair_corr >= all_w1_selection_threshold:
            keep = False
            break

    if keep:
        all_w1_selected.append(feature)

    all_w1_trace_rows.append(
        {
            "rank": int(row["rank"]),
            "metric": row["metric"],
            "feature": feature,
            "mean_abs_corr": float(row["mean_abs_corr"]),
            "corr_2022": float(row["corr_2022"]),
            "corr_2024": float(row["corr_2024"]),
            "decision": "keep" if keep else "drop as too similar",
            "closest_selected": closest_selected,
            "abs_pair_corr_to_selected": abs_pair_corr,
        }
    )

all_w1_selection_trace = pd.DataFrame(all_w1_trace_rows)

all_w1_shortlist = all_w1_ranked[all_w1_ranked["feature"].isin(all_w1_selected)].copy()
all_w1_shortlist["selection_threshold"] = all_w1_selection_threshold

all_w1_shortlist_metric_counts = (
    all_w1_shortlist.groupby("metric", as_index=False)
    .agg(selected_features=("feature", "size"))
)

all_w1_pair_rows = []
for i, left in enumerate(all_w1_selected):
    for right in all_w1_selected[i + 1:]:
        valid = all_w1_features[left].notna() & all_w1_features[right].notna()
        if valid.sum() < 30:
            continue
        all_w1_pair_rows.append(
            {
                "left_feature": left,
                "right_feature": right,
                "pair_corr": float(all_w1_features[left][valid].corr(all_w1_features[right][valid])),
            }
        )

all_w1_shortlist_pairs = pd.DataFrame(all_w1_pair_rows)

display(all_w1_ranked.round(3))
display(all_w1_same_sign.round(3))
display(all_w1_selection_trace.round(3))
display(all_w1_shortlist[["rank", "metric", "feature", "mean_abs_corr", "corr_2022", "corr_2024", "selection_threshold"]].round(3))
display(all_w1_shortlist_metric_counts)
display(all_w1_shortlist_pairs.round(3))

fig, ax = plt.subplots(figsize=(10, 8))
plot_all_w1 = all_w1_ranked.sort_values("mean_abs_corr", ascending=True)
colors = ["#5b8a72" if feature in all_w1_selected else "#c8c8c8" for feature in plot_all_w1["feature"]]
ax.barh(plot_all_w1["feature"], plot_all_w1["mean_abs_corr"], color=colors)
ax.set_title("all w1 media variables ranking and final shortlist")
ax.set_xlabel("mean absolute correlation with gov margin")
ax.grid(axis="x")
plt.tight_layout()
plt.show()
# clean up temporary variables, keep only the all w1 shortlist objects
for var in [
    "abs_corrs",
    "abs_pair_corr",
    "all_w1_features",
    "all_w1_pair_rows",
    "all_w1_prefix_specs",
    "all_w1_rows",
    "all_w1_selection_threshold",
    "all_w1_trace_rows",
    "all_w1_transform_rule",
    "ax",
    "chosen",
    "closest_selected",
    "col",
    "colors",
    "corr",
    "feature",
    "fig",
    "i",
    "keep",
    "left",
    "metric",
    "metric_cols",
    "ok",
    "outlet",
    "pair_corr",
    "plot_all_w1",
    "prefix",
    "right",
    "row",
    "same_sign",
    "sign_now",
    "sign_ref",
    "sub",
    "transformed_full",
    "transformed_year",
    "valid",
    "year",
    "year_corr",
]:
    if var in locals():
        del locals()[var]
del var


### Easy English interpretation of the all-`w1` shortlist block

This is the broadest `w1` check in the notebook.

How to read it:

- The first table is the full ranking of all `w1` candidates from `real_users`, `views`, and `time_per_view`.
- A high `mean_abs_corr` still matters here. It is the first screening score.
- The second table keeps only variables with the same sign in both usable election years.
- The third table shows the greedy redundancy step.
- The shortlist table is the final result after all three checks.
- The metric-count table shows which media type survives more often after the redundancy filter.

The important warning is this:

If a weaker variable survives only because it is less correlated with the already selected variables, that does not automatically make it substantively better than a stronger variable.
So the final shortlist should be read together with the ranking table, not instead of the ranking table.


### Helper for the repeated all-window shortlist checks

The next two blocks repeat the same final shortlist logic for `w2` and `w3`.
They use the same three steps as the broad `w1` block:

- rank variables by `mean_abs_corr`
- keep only variables with the same sign in both years
- drop variables that are too similar to an already selected variable


In [ ]:
# this block defines helper functions that later cells reuse.
def run_window_shortlist(window_label, threshold=0.90):
    prefix_specs = {
        "real_users": f"real_users_{window_label}_",
        "views": f"views_{window_label}_",
        "time_per_view_s": f"time_per_view_s_{window_label}_",
    }
    transform_rule = {
        "real_users": "per_1000_log",
        "views": "per_1000_log",
        "time_per_view_s": "log1p",
    }

    rows = []
    features = {}

    for metric, prefix in prefix_specs.items():
        metric_cols = [c for c in media_panel.columns if c.startswith(prefix)]

        for col in metric_cols:
            outlet = col[len(prefix):]
            feature_name = f"{metric} {outlet}"
            transformed_full = transform_media(media_panel[col], media_panel["pop_total"], transform_rule[metric])
            features[feature_name] = transformed_full

            year_corr = {}
            abs_corrs = []
            same_sign = True
            sign_ref = None
            ok = True

            for year in [2022, 2024]:
                sub = media_panel[media_panel["year"] == year].copy()
                transformed_year = transform_media(sub[col], sub["pop_total"], transform_rule[metric])
                valid = transformed_year.notna() & sub["gov_margin"].notna()

                if valid.sum() < 30:
                    ok = False
                    break
                if transformed_year[valid].std() == 0:
                    ok = False
                    break

                corr = float(transformed_year[valid].corr(sub.loc[valid, "gov_margin"]))
                year_corr[year] = corr
                abs_corrs.append(abs(corr))

                sign_now = np.sign(corr)
                if sign_ref is None:
                    sign_ref = sign_now
                elif sign_now != sign_ref:
                    same_sign = False

            if not ok:
                continue

            rows.append(
                {
                    "metric": metric,
                    "outlet": outlet,
                    "feature": feature_name,
                    "corr_2022": year_corr[2022],
                    "corr_2024": year_corr[2024],
                    "same_sign": same_sign,
                    "mean_abs_corr": float(np.mean(abs_corrs)),
                }
            )

    ranked = pd.DataFrame(rows).sort_values("mean_abs_corr", ascending=False).reset_index(drop=True)
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked = ranked[["rank", "metric", "outlet", "feature", "corr_2022", "corr_2024", "same_sign", "mean_abs_corr"]]

    same_sign_ranked = ranked[ranked["same_sign"]].reset_index(drop=True).copy()

    selected = []
    trace_rows = []

    for _, row in same_sign_ranked.iterrows():
        feature = row["feature"]
        keep = True
        closest_selected = pd.NA
        abs_pair_corr = pd.NA

        for chosen in selected:
            valid = features[feature].notna() & features[chosen].notna()
            if valid.sum() < 30:
                continue

            pair_corr = abs(float(features[feature][valid].corr(features[chosen][valid])))

            if pd.isna(abs_pair_corr) or pair_corr > abs_pair_corr:
                abs_pair_corr = pair_corr
                closest_selected = chosen

            if pair_corr >= threshold:
                keep = False
                break

        if keep:
            selected.append(feature)

        trace_rows.append(
            {
                "rank": int(row["rank"]),
                "metric": row["metric"],
                "feature": feature,
                "mean_abs_corr": float(row["mean_abs_corr"]),
                "corr_2022": float(row["corr_2022"]),
                "corr_2024": float(row["corr_2024"]),
                "decision": "keep" if keep else "drop as too similar",
                "closest_selected": closest_selected,
                "abs_pair_corr_to_selected": abs_pair_corr,
            }
        )

    selection_trace = pd.DataFrame(trace_rows)
    shortlist = ranked[ranked["feature"].isin(selected)].copy()
    shortlist["selection_threshold"] = threshold

    shortlist_metric_counts = (
        shortlist.groupby("metric", as_index=False)
        .agg(selected_features=("feature", "size"))
    )

    pair_rows = []
    for i, left in enumerate(selected):
        for right in selected[i + 1:]:
            valid = features[left].notna() & features[right].notna()
            if valid.sum() < 30:
                continue
            pair_rows.append(
                {
                    "left_feature": left,
                    "right_feature": right,
                    "pair_corr": float(features[left][valid].corr(features[right][valid])),
                }
            )

    shortlist_pairs = pd.DataFrame(pair_rows)

    display(ranked.round(3))
    display(same_sign_ranked.round(3))
    display(selection_trace.round(3))
    display(shortlist[["rank", "metric", "feature", "mean_abs_corr", "corr_2022", "corr_2024", "selection_threshold"]].round(3))
    display(shortlist_metric_counts)
    display(shortlist_pairs.round(3))

    fig, ax = plt.subplots(figsize=(10, 8))
    plot_ranked = ranked.sort_values("mean_abs_corr", ascending=True)
    colors = ["#5b8a72" if feature in selected else "#c8c8c8" for feature in plot_ranked["feature"]]
    ax.barh(plot_ranked["feature"], plot_ranked["mean_abs_corr"], color=colors)
    ax.set_title(f"all {window_label} media variables ranking and final shortlist")
    ax.set_xlabel("mean absolute correlation with gov margin")
    ax.grid(axis="x")
    plt.tight_layout()
    plt.show()

    return {
        "ranked": ranked,
        "same_sign": same_sign_ranked,
        "selection_trace": selection_trace,
        "shortlist": shortlist,
        "shortlist_metric_counts": shortlist_metric_counts,
        "shortlist_pairs": shortlist_pairs,
        "selected": selected,
    }


Interpretation. The variables that remain here are the safer candidates for the later modelling steps.


### Step by step shortlist with all `w2` media variables

This repeats the final broad shortlist check, but now only for `w2` media variables.
So this block compares `real_users_w2`, `views_w2`, and `time_per_view_s_w2` variables.


In [ ]:
# this block builds or checks a smaller shortlist of variables for later modelling.
all_w2_results = run_window_shortlist("w2")


### Easy English interpretation of the all-`w2` shortlist block

This block has the same logic as the broad `w1` block.

How to read it:

- The first table is the full ranking of all `w2` candidates from `real_users`, `views`, and `time_per_view`.
- The first screen is still `mean_abs_corr`, so a higher value is still better as a starting signal.
- The second table keeps only variables with the same sign in both usable election years.
- The third table shows which variables were dropped because they were too similar to a stronger selected variable.
- The shortlist table is the final `w2` result after all three checks.
- The metric-count table helps show which media type survives the redundancy filter more often.

The same warning still applies:

A variable can survive because it is less redundant, even if it is weaker than another variable in raw correlation.
So the shortlist should always be read together with the ranking table.


### Step by step shortlist with all `w3` media variables

This repeats the same final shortlist check for `w3` media variables.
So this block compares `real_users_w3`, `views_w3`, and `time_per_view_s_w3` variables.


In [ ]:
# this block builds or checks a smaller shortlist of variables for later modelling.
all_w3_results = run_window_shortlist("w3")


### Easy English interpretation of the all-`w3` shortlist block

This block has the same logic as the `w1` and `w2` shortlist checks.

How to read it:

- The first table is the full ranking of all `w3` candidates from `real_users`, `views`, and `time_per_view`.
- The first screen is still `mean_abs_corr`, so a higher value is still better as a starting signal.
- The second table keeps only variables with the same sign in both usable election years.
- The third table shows which variables were dropped because they were too similar to an already selected variable.
- The shortlist table is the final `w3` result after all three checks.
- The metric-count table helps show which media type survives the redundancy filter more often.

The same warning still applies here too:

A variable can survive because it is less redundant, even if it is weaker than another variable in raw correlation.
So the shortlist should always be read together with the ranking table.


### Manual seven-outlet media shortlist for correlation inspection

Based on the full exploratory checks, the most defensible first choice is to use only `w1 real_users` variables.

Why this is the preferred choice:

- `real_users` gave the strongest and most stable signal across the media checks.
- `views` was sometimes useful, but usually weaker than `real_users`.
- `time_per_view` was much weaker on average, and the broad shortlist often kept many of these variables only because they were less redundant.
- `w3` is the closest pre-election media exposure measure in the actual window construction.
- `w1`, `w2`, and `w3` were very similar overall, so using one window is simpler and reduces overfitting risk.

The manual shortlist for correlation inspection is:

- `real_users_w1_tenyek_b`
- `real_users_w1_mediaklikk_b`
- `real_users_w1_telex_b`
- `real_users_w1_rtl_b`
- `real_users_w1_hvg_b`
- `real_users_w1_444_b`
- `real_users_w1_origo_b`

Why this is a sensible manual choice:

- it is easy to interpret politically and substantively
- it covers different parts of the media space without bringing in every outlet
- it stays small enough for an initial exploratory comparison
- the next block lets us see directly how redundant these seven outlets are with each other

### Redundancy check for the manual seven-outlet shortlist

This block checks how strongly the manual `w1 real_users` shortlist moves together.

The shortlist is:

- `real_users_w1_tenyek_b`
- `real_users_w1_mediaklikk_b`
- `real_users_w1_telex_b`
- `real_users_w1_rtl_b`
- `real_users_w1_hvg_b`
- `real_users_w1_444_b`
- `real_users_w1_origo_b`

How to read the output:

- The correlation matrix shows the signed pairwise correlations.
- The absolute-correlation matrix shows only the strength of the shared movement.
- Values close to `1` mean the two variables move very similarly in the usable sample.
- If many values are very high, the shortlist is still partly redundant even if it is substantively more balanced.

In [ ]:
# this block builds or checks a smaller shortlist of variables for later modelling.
final_manual_outlets = ["tenyek_b", "mediaklikk_b", "telex_b", "rtl_b", "hvg_b", "444_b", "origo_b"]
final_manual_cols = [f"real_users_w1_{outlet}" for outlet in final_manual_outlets]

final_manual_sub = media_panel[media_panel["year"].isin([2022, 2024])].copy()
final_manual_features = pd.DataFrame(
    {
        outlet: transform_media(final_manual_sub[col], final_manual_sub["pop_total"], "per_1000_log")
        for outlet, col in zip(final_manual_outlets, final_manual_cols)
    }
)

final_manual_valid = final_manual_features.notna().all(axis=1) & final_manual_sub["gov_margin"].notna()
final_manual_features_valid = final_manual_features.loc[final_manual_valid].copy()

final_manual_corr = final_manual_features_valid.corr()
final_manual_abs_corr = final_manual_corr.abs()

final_manual_pair_rows = []
for i, left in enumerate(final_manual_outlets):
    for right in final_manual_outlets[i + 1:]:
        final_manual_pair_rows.append(
            {
                "left_outlet": left,
                "right_outlet": right,
                "pair_corr": float(final_manual_corr.loc[left, right]),
                "abs_pair_corr": float(final_manual_abs_corr.loc[left, right]),
            }
        )

final_manual_pairs = pd.DataFrame(final_manual_pair_rows).sort_values("abs_pair_corr", ascending=False).reset_index(drop=True)

display(final_manual_corr.round(3))
display(final_manual_abs_corr.round(3))
display(final_manual_pairs.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(final_manual_corr.values, cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set_title("final manual shortlist correlation")
axes[0].set_xticks(range(len(final_manual_outlets)))
axes[0].set_yticks(range(len(final_manual_outlets)))
axes[0].set_xticklabels(final_manual_outlets, rotation=45, ha="right")
axes[0].set_yticklabels(final_manual_outlets)
for i in range(len(final_manual_outlets)):
    for j in range(len(final_manual_outlets)):
        axes[0].text(j, i, f"{final_manual_corr.values[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)

im1 = axes[1].imshow(final_manual_abs_corr.values, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("final manual shortlist absolute correlation")
axes[1].set_xticks(range(len(final_manual_outlets)))
axes[1].set_yticks(range(len(final_manual_outlets)))
axes[1].set_xticklabels(final_manual_outlets, rotation=45, ha="right")
axes[1].set_yticklabels(final_manual_outlets)
for i in range(len(final_manual_outlets)):
    for j in range(len(final_manual_outlets)):
        axes[1].text(j, i, f"{final_manual_abs_corr.values[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)

fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
# clean up temporary variables, keep only the final manual shortlist outputs
for var in [
    "axes",
    "fig",
    "final_manual_cols",
    "final_manual_features",
    "final_manual_features_valid",
    "final_manual_pair_rows",
    "final_manual_sub",
    "final_manual_valid",
    "i",
    "im0",
    "im1",
    "j",
    "left",
    "right",
]:
    if var in locals():
        del locals()[var]
del var


Interpretation. The variables that remain here are the safer candidates for the later modelling steps.


### Downstream media decision after the redundancy check

The redundancy checks above are still useful for inspection.

The next code cell is now the editable downstream setup.

You can change these in one place:

- `downstream_media_windows`, for example `["w1"]` or `["w1", "w2", "w3"]`
- `downstream_media_metrics`, for example `["real_users"]`, `["views"]`, `["time_per_view_s"]`, or a mix
- `requested_media_names`, if you want a shorter or different outlet list

The block keeps only requested outlet, metric, and window combinations that really exist in the dataset.


In [ ]:
# Editable downstream setup
# Change only these lists when you want to test another media setup.
downstream_media_windows = ["w3"]
downstream_media_metrics = ["real_users"]

downstream_transform_rule = {
    "real_users": "per_1000_log",
    "views": "per_1000_log",
    "time_per_view_s": "log1p",
}

requested_media_names = [
    "24 Óra",
    "24ora.hu",
    "888.hu",
    "BORS",
    "BORSONLINE",
    "Bartók",
    "Blogstar",
    "Békés Megyei Hírlap",
    "Chili TV",
    "Dankó",
    "Demokrata",
    "Demokrata.hu",
    "Duna",
    "Duna World",
    "Dunaújvárosi Hírlap",
    "Dunaújvárosi Hírlap online",
    "Délmagyar.hu",
    "Délmagyarország",
    "ESMA",
    "Fejér Megyei Hírlap",
    "Fem3",
    "Feol.hu",
    "Figyelo.hu",
    "Figyelő",
    "Gong Rádió",
    "Hajdu-Bihari Napló",
    "Hello!",
    "Heves Megyei Hírlap",
    "HírTV",
    "Híradó.hu",
    "IzauraTV",
    "Karc FM",
    "Kelet-Magyarország",
    "Kisalföld",
    "Kiwi TV",
    "Komárom-Esztergom megyei 24 óra",
    "Kossuth",
    "Life",
    "Life TV",
    "Lokál",
    "Lokál Extra",
    "Lokál.hu",
    "M1",
    "M2",
    "M3",
    "M4",
    "M4Sport.hu",
    "M5",
    "MANAGER MAGAZIN",
    "MNO",
    "MSN Mai Nap",
    "MTI",
    "Magyar Hírlap",
    "Magyar Hírlap Online",
    "Magyar Nemzet",
    "Mahir Cityposter",
    "Mandiner",
    "Mediaklikk.hu",
    "Mozi+",
    "Napló",
    "Nemzeti Sport",
    "Nemzeti Sport Online",
    "Nemzetiségi Rádió",
    "Newsfeed",
    "Nógrád Megyei Hírlap",
    "Origo",
    "Parlamenti Rádió",
    "Pestisrácok.hu",
    "Petőfi",
    "Petőfi Népe",
    "Prime",
    "Publimont",
    "Retro Rádió",
    "Ripost",
    "Ripost.hu",
    "Rádió1",
    "Somogyi Hírlap",
    "Spíler",
    "SuperTV2",
    "Szabad Föld",
    "Szegedma",
    "TV2",
    "TV2.hu",
    "Tolnai Népújság",
    "Tények.hu",
    "VAS NÉPE",
    "Vas Népe online",
    "Vasárnap Reggel",
    "Világgazdaság",
    "Zalai Hírlap",
    "petofinepe.hu",
    "sonline.hu",
    "szoljon.hu",
    "szon.hu",
    "teol.hu",
    "ujneplap.hu",
    "vg.hu",
    "Észak-Magyarország",
    "Új Dunántúli Napló",
    "Új Néplap",
]

requested_to_dataset_outlet = {
    "24 Óra": "24_b",
    "24ora.hu": "24_b",
    "Komárom-Esztergom megyei 24 óra": "24_b",
    "BORS": "borsonline_b",
    "BORSONLINE": "borsonline_b",
    "Híradó.hu": "hirado_b",
    "MNO": "magyarnemzet_b",
    "Magyar Nemzet": "magyarnemzet_b",
    "Mandiner": "mandiner_b",
    "Mediaklikk.hu": "mediaklikk_b",
    "Origo": "origo_b",
    "Ripost": "ripost_b",
    "Ripost.hu": "ripost_b",
    "TV2": "tv2_b",
    "TV2.hu": "tv2_b",
    "Tények.hu": "tenyek_b",
}

valid_windows = {"w1", "w2", "w3"}
downstream_media_windows = [window for window in downstream_media_windows if window in valid_windows]
downstream_media_metrics = [
    metric for metric in downstream_media_metrics
    if metric in downstream_transform_rule
]

requested_media_rows = []
selected_requested_media_outlets = []

for requested_name in requested_media_names:
    dataset_outlet = requested_to_dataset_outlet.get(requested_name, pd.NA)
    available_raw_cols = []

    if pd.notna(dataset_outlet):
        for metric in downstream_media_metrics:
            for window in downstream_media_windows:
                raw_col = f"{metric}_{window}_{dataset_outlet}"
                if raw_col in final_statistics_oevk.columns:
                    available_raw_cols.append(raw_col)

    requested_media_rows.append(
        {
            "requested_name": requested_name,
            "dataset_outlet": dataset_outlet,
            "available_raw_cols": " | ".join(available_raw_cols) if available_raw_cols else pd.NA,
            "available_in_dataset": bool(available_raw_cols),
        }
    )

    if available_raw_cols and dataset_outlet not in selected_requested_media_outlets:
        selected_requested_media_outlets.append(dataset_outlet)

requested_media_lookup = pd.DataFrame(requested_media_rows)

if selected_requested_media_outlets:
    selected_requested_outlet_table = (
        requested_media_lookup[requested_media_lookup["available_in_dataset"]]
        .groupby("dataset_outlet", as_index=False)
        .agg(
            requested_names=("requested_name", lambda values: " | ".join(values)),
            available_raw_cols=("available_raw_cols", lambda values: " | ".join(sorted(set(values)))),
        )
    )
else:
    selected_requested_outlet_table = pd.DataFrame(
        columns=["dataset_outlet", "requested_names", "available_raw_cols"]
    )

selected_requested_media_specs = []
for outlet in selected_requested_media_outlets:
    requested_names = selected_requested_outlet_table.loc[
        selected_requested_outlet_table["dataset_outlet"] == outlet,
        "requested_names",
    ].iloc[0]

    for metric in downstream_media_metrics:
        for window in downstream_media_windows:
            raw_col = f"{metric}_{window}_{outlet}"
            if raw_col not in final_statistics_oevk.columns:
                continue

            selected_requested_media_specs.append(
                {
                    "dataset_outlet": outlet,
                    "requested_names": requested_names,
                    "metric": metric,
                    "window": window,
                    "raw_col": raw_col,
                    "feature": f"media_{window}_{metric}_{outlet}",
                    "transform_rule": downstream_transform_rule[metric],
                }
            )

selected_requested_media_specs_table = pd.DataFrame(selected_requested_media_specs)
selected_requested_media_features = (
    selected_requested_media_specs_table["feature"].tolist()
    if len(selected_requested_media_specs_table)
    else []
)

downstream_media_config = pd.DataFrame(
    [
        {
            "setting": "windows",
            "value": " | ".join(downstream_media_windows) if downstream_media_windows else "none",
        },
        {
            "setting": "metrics",
            "value": " | ".join(downstream_media_metrics) if downstream_media_metrics else "none",
        },
    ]
)

requested_media_summary = pd.DataFrame(
    [
        {
            "scenario": "requested_names_total",
            "count": len(requested_media_names),
            "note": "full requested outlet list from your instruction",
        },
        {
            "scenario": "names_with_dataset_mapping",
            "count": int(requested_media_lookup["dataset_outlet"].notna().sum()),
            "note": "requested names that can be mapped to a dataset outlet name",
        },
        {
            "scenario": "available_unique_dataset_outlets",
            "count": len(selected_requested_media_outlets),
            "note": "unique requested outlets that match at least one chosen window and metric",
        },
        {
            "scenario": "available_selected_features",
            "count": len(selected_requested_media_features),
            "note": "total selected feature combinations after the window and metric filter",
        },
    ]
)

display(downstream_media_config)
display(requested_media_lookup)
display(selected_requested_outlet_table)
display(selected_requested_media_specs_table)
display(requested_media_summary)

selected_requested_media_specs_table
# clean up temporary variables, keep the selected downstream media setup
for var in [
    "available_raw_cols",
    "dataset_outlet",
    "downstream_media_config",
    "metric",
    "outlet",
    "raw_col",
    "requested_media_lookup",
    "requested_media_rows",
    "requested_media_summary",
    "requested_name",
    "requested_names",
    "selected_requested_outlet_table",
    "valid_windows",
    "window",
]:
    if var in locals():
        del locals()[var]
del var


### Missingness check for the editable downstream media setup

This block checks the selected requested media features after the editable window and metric setup above.

It asks two practical questions:

- how complete is each selected feature by year
- and how many rows survive if all selected features must be present together

Interpretation:

- `keep_full_history` = enough data in all historical years and in `2026`
- `recent_only` = enough data only in the recent years
- `drop_for_no_imputation` = too much data is missing


In [ ]:
# this block checks missingness so we can see which predictors are safe to use later.
strict_coverage_threshold = 0.95
full_history_years = [2018, 2019, 2022, 2024]
recent_years = [2022, 2024, 2026]

media_missingness_source = final_statistics_oevk.copy()
media_missingness_source["pop_total"] = pd.to_numeric(
    media_missingness_source["pop_total"],
    errors="coerce",
).replace({0: np.nan})

selected_media_feature_rows = []

for spec in selected_requested_media_specs:
    media_missingness_source[spec["feature"]] = transform_media(
        media_missingness_source[spec["raw_col"]],
        media_missingness_source["pop_total"],
        spec["transform_rule"],
    )

    per_year_coverage = {}
    for year in full_history_years + [2026]:
        sub = media_missingness_source[media_missingness_source["year"] == year]
        per_year_coverage[year] = float(sub[spec["feature"]].notna().mean()) if len(sub) else np.nan

    full_history_values = [
        per_year_coverage[year]
        for year in full_history_years
        if pd.notna(per_year_coverage[year])
    ]
    recent_values = [
        per_year_coverage[year]
        for year in recent_years
        if pd.notna(per_year_coverage[year])
    ]

    if full_history_values and min(full_history_values) >= strict_coverage_threshold and per_year_coverage[2026] >= strict_coverage_threshold:
        recommendation = "keep_full_history"
    elif recent_values and min(recent_values) >= strict_coverage_threshold:
        recommendation = "recent_only"
    else:
        recommendation = "drop_for_no_imputation"

    selected_media_feature_rows.append(
        {
            "dataset_outlet": spec["dataset_outlet"],
            "requested_names": spec["requested_names"],
            "metric": spec["metric"],
            "window": spec["window"],
            "raw_col": spec["raw_col"],
            "feature": spec["feature"],
            "nonmissing_2018": per_year_coverage[2018],
            "nonmissing_2019": per_year_coverage[2019],
            "nonmissing_2022": per_year_coverage[2022],
            "nonmissing_2024": per_year_coverage[2024],
            "nonmissing_2026": per_year_coverage[2026],
            "recommendation": recommendation,
        }
    )

selected_media_feature_missingness = pd.DataFrame(selected_media_feature_rows)

if selected_requested_media_features:
    selected_media_complete_case_rows = []

    for year in full_history_years + [2026]:
        sub = media_missingness_source[media_missingness_source["year"] == year].copy()
        complete_mask = sub[selected_requested_media_features].notna().all(axis=1)
        any_mask = sub[selected_requested_media_features].notna().any(axis=1)

        selected_media_complete_case_rows.append(
            {
                "year": year,
                "selected_feature_count": len(selected_requested_media_features),
                "complete_rows": int(complete_mask.sum()),
                "complete_case_share": float(complete_mask.mean()) if len(sub) else np.nan,
                "rows_with_any_selected_media": int(any_mask.sum()),
                "share_with_any_selected_media": float(any_mask.mean()) if len(sub) else np.nan,
            }
        )

    selected_media_complete_case = pd.DataFrame(selected_media_complete_case_rows)
else:
    selected_media_complete_case = pd.DataFrame(
        columns=[
            "year",
            "selected_feature_count",
            "complete_rows",
            "complete_case_share",
            "rows_with_any_selected_media",
            "share_with_any_selected_media",
        ]
    )

selected_media_na_summary = pd.DataFrame(
    [
        {
            "scenario": "available_selected_features",
            "count": len(selected_requested_media_features),
            "note": "selected feature combinations that exist in the dataset",
        },
        {
            "scenario": "keep_full_history_features",
            "count": int((selected_media_feature_missingness["recommendation"] == "keep_full_history").sum()) if len(selected_media_feature_missingness) else 0,
            "note": "selected features with very strong coverage in all years",
        },
        {
            "scenario": "recent_only_features",
            "count": int((selected_media_feature_missingness["recommendation"] == "recent_only").sum()) if len(selected_media_feature_missingness) else 0,
            "note": "selected features that are usable only as recent year extensions",
        },
        {
            "scenario": "drop_for_no_imputation_features",
            "count": int((selected_media_feature_missingness["recommendation"] == "drop_for_no_imputation").sum()) if len(selected_media_feature_missingness) else 0,
            "note": "selected features that still have too much missingness",
        },
    ]
)

display(selected_media_feature_missingness.round(3))
display(selected_media_complete_case.round(3))
display(selected_media_na_summary)
# clean up temporary variables, keep the selected media missingness outputs
for var in [
    "any_mask",
    "complete_mask",
    "full_history_values",
    "per_year_coverage",
    "recent_values",
    "recommendation",
    "selected_media_complete_case_rows",
    "selected_media_feature_rows",
    "spec",
    "sub",
    "year",
]:
    if var in locals():
        del locals()[var]
del var


### Easy English interpretation of the requested-media missingness output

This output now checks an editable multi-outlet media block.

Change the window and metric lists in the previous code cell if you want to test another setup.

**Table 1: selected feature coverage**

- each row is one selected outlet, metric, and window combination that really exists in the dataset
- `recommendation` shows whether that feature is safe for full-history use, recent-only use, or should be dropped in a strict no-imputation setup

**Table 2: complete-case check for the full selected set**

- `complete_case_share` shows how many rows stay usable if all selected features must be present together
- `share_with_any_selected_media` shows how many rows have at least one selected feature available

So the key tradeoff is simple.

- adding more windows or metrics makes the media block richer
- but it can also shrink the usable sample fast if the full selected set has weak joint coverage


### Aggregated `pro_gov_avg` feature

This final block builds one simple aggregated recent media feature from the currently selected `w3` media variables.

The rule is simple:

- first transform each selected `w3` media variable with the same log per-1,000-resident rule used earlier
- then take the row-wise average across those transformed selected variables

So `pro_gov_avg` is one recent pro-government media summary per OEVK.


In [ ]:
# this block builds the aggregated pro government media feature used later.
# keep downstream_media_metrics to one metric family here; mixing real_users, views,
# and time_per_view_s would make pro_gov_avg harder to interpret.
pro_gov_feature_specs = [
    spec for spec in selected_requested_media_specs
    if spec["window"] == "w3"
]
pro_gov_feature_cols = [spec["feature"] for spec in pro_gov_feature_specs]

if not pro_gov_feature_cols:
    raise ValueError("No selected w3 media features were found for pro_gov_avg.")

pro_gov_avg_source = media_missingness_source.copy()
pro_gov_avg_source["pro_gov_avg"] = pro_gov_avg_source[pro_gov_feature_cols].mean(axis=1)

pro_gov_avg_lookup = pro_gov_avg_source[["year", "oevk_id", "pro_gov_avg"]].copy()

final_statistics_oevk = final_statistics_oevk.drop(columns=["pro_gov_avg"], errors="ignore").merge(
    pro_gov_avg_lookup,
    on=["year", "oevk_id"],
    how="left",
)

media_panel = media_panel.drop(columns=["pro_gov_avg"], errors="ignore").merge(
    pro_gov_avg_lookup,
    on=["year", "oevk_id"],
    how="left",
)

pro_gov_avg_summary = (
    pro_gov_avg_source.groupby("year", as_index=False)
    .agg(
        rows=("oevk_id", "size"),
        avg_pro_gov_avg=("pro_gov_avg", "mean"),
        pro_gov_avg_nonmissing_share=("pro_gov_avg", lambda s: float(s.notna().mean())),
    )
)

pro_gov_raw_cols = [spec["raw_col"] for spec in pro_gov_feature_specs]
pro_gov_media_outlets = list(dict.fromkeys(spec["dataset_outlet"] for spec in pro_gov_feature_specs))
downstream_recent_media_config = {
    "selected_single_media_outlet": "pro_gov_avg",
    "selected_media_window": "w3",
    "selected_media_metric": "real_users",
    "selected_media_outlets": pro_gov_media_outlets,
    "selected_media_raw_cols": pro_gov_raw_cols,
    "selected_media_feature_cols": ["pro_gov_avg"],
    "selected_recent_media_features": ["pro_gov_avg"],
    "aggregation_method": "mean_of_selected_w3_features",
}

import json

screened_feature_output_path = project_root / "data" / "created" / "screened_feature_sets.json"
if screened_feature_output_path.exists():
    screened_feature_export = json.loads(screened_feature_output_path.read_text(encoding="utf-8"))
else:
    screened_feature_export = {}

legacy_media_config_keys = [
    "screened_recent_media_extension",
    "selected_single_media_outlet",
]
for legacy_media_config_key in legacy_media_config_keys:
    screened_feature_export.pop(legacy_media_config_key, None)

screened_feature_export["downstream_recent_media_config"] = downstream_recent_media_config
screened_feature_output_path.parent.mkdir(parents=True, exist_ok=True)
screened_feature_output_path.write_text(
    json.dumps(screened_feature_export, indent=2),
    encoding="utf-8",
)

print("Saved downstream media config to:")
print(screened_feature_output_path)

display(pd.DataFrame({"w3_selected_feature": pro_gov_feature_cols}))
display(pro_gov_avg_summary.round(3))
display(
    final_statistics_oevk[["year", "oevk_id", "county", "pro_gov_avg"]]
    .dropna(subset=["pro_gov_avg"])
    .head(10)
    .round(3)
)
# clean up temporary variables, keep final_statistics_oevk with pro_gov_avg
for var in [
    "downstream_recent_media_config",
    "legacy_media_config_keys",
    "pro_gov_media_outlets",
    "pro_gov_raw_cols",
    "pro_gov_avg_lookup",
    "pro_gov_avg_source",
    "pro_gov_feature_cols",
    "pro_gov_feature_specs",
    "screened_feature_export",
    "screened_feature_output_path",
]:
    if var in locals():
        del locals()[var]
del var


Interpretation. Lower coverage means the variable is harder to use safely without imputation.


### Log Transformation Rationale

This section visualizes why the log transformation $\log(1 + \text{raw metric} / \text{pop\_total})$ is necessary for media audience data. Raw audience ratios are typically highly skewed with a long right tail, which can lead to disproportionate influence of high-reach outliers in linear models. The log transformation compresses the scale and results in a more balanced, near-normal distribution suitable for Ridge regression.

In [ ]:
# this block shows why the transformed media feature is easier to work with.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# set global font to Times New Roman, size 12
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 12

# picking the aggregated pro-government average for demonstration
raw_metric_col = "pro_gov_avg"

if raw_metric_col in final_statistics_oevk.columns:
    # to show the rationale, we compare a typical raw component ratio to the final log-average predictor
    example_raw = "real_users_w1_mediaklikk_b"  # one of the components of pro_gov_avg

    raw_values = pd.to_numeric(final_statistics_oevk[example_raw], errors="coerce").dropna()
    pop_total = pd.to_numeric(final_statistics_oevk["pop_total"], errors="coerce").loc[raw_values.index]
    raw_ratio = raw_values / pop_total

    # plotting: two histograms side by side for comparison
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=False, sharey=False)

    # left plot: distribution of raw ratios (likely highly skewed)
    axes[0].hist(raw_ratio, bins=30, color="#edc948", edgecolor="white")
    axes[0].set_title(f"Distribution of Raw Ratio", fontsize=12)
    axes[0].set_xlabel("Raw ratio (raw count / total population)", fontsize=12)
    axes[0].set_ylabel("Number of observations", fontsize=12)
    axes[0].grid(axis="y", alpha=0.3)

    # right plot: distribution of the final pro_gov_avg predictor
    final_vals = pd.to_numeric(final_statistics_oevk[raw_metric_col], errors="coerce").dropna()
    axes[1].hist(final_vals, bins=30, color="#e15759", edgecolor="white")
    axes[1].set_title(f"Distribution of the Final Predictor", fontsize=12)
    axes[1].set_xlabel("Log-transformed weighted average value", fontsize=12)
    axes[1].set_ylabel("Number of observations", fontsize=12)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()

    # save the figure to both requested folders
    if "project_root" not in locals():
        try:
            here = Path(__file__).resolve().parent
        except NameError:
            here = Path.cwd()

        project_root = here
        while not (project_root / "data").exists() and project_root != project_root.parent:
            project_root = project_root.parent

    output_fig_paths = [
        project_root / "graphs_tables" / "raw_vs_final_media_predictor_distribution.png",
        project_root / "doc" / "ÚJLaTex_ENGLISH_Template másolat 2" / "figures" / "raw_vs_final_media_predictor_distribution.png",
    ]
    for output_fig_path in output_fig_paths:
        output_fig_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_fig_path, dpi=300, bbox_inches="tight")

    plt.show()

    print("Saved figure to:")
    for output_fig_path in output_fig_paths:
        print(output_fig_path)

else:
    print(f"The column '{raw_metric_col}' was not found. Please ensure the cells above have been executed.")

# clean up
for var in [
    "raw_metric_col", "example_raw", "raw_values", "pop_total", "raw_ratio",
    "final_vals", "fig", "axes", "output_fig_paths", "output_fig_path", "here"
]:
    if var in locals():
        del locals()[var]
del var


Interpretation. The transformed version is easier to model because extreme values matter less.


### Final Correlation Visualization (W3)

This block visualizes the correlation of real user reach (W3) with the governmental margin across all media outlets. Pro-government outlets are highlighted in orange, others in grey, and the aggregated `pro_gov_avg` is included in red for reference.

In [ ]:
# this block checks correlations so we can spot variables that overlap too much.
from pathlib import Path
from matplotlib.patches import Patch

all_w3_prefix_specs = {
    "real_users": "real_users_w3_",
}
all_w3_transform_rule = {
    "real_users": "per_1000_log",
}

pro_gov_outlets = ["origo", "tv2", "hirado", "mediaklikk", "mandiner", "magyarnemzet", "ripost", "tenyek", "borsonline", "index"]

all_w3_rows = []

for metric, prefix in all_w3_prefix_specs.items():
    # Use final_statistics_oevk to ensure we have 2026 data
    metric_cols = [c for c in final_statistics_oevk.columns if c.startswith(prefix)]

    for col in metric_cols:
        outlet = col[len(prefix):]
        if outlet == "avg":
            continue

        # Nice naming logic matching the LaTeX table requirement
        nice_name = outlet.replace("_b", "").replace("_com", "").replace("_hu", "").replace("_app", "").capitalize()

        # Calculate the simple log-transformed value for 2026
        mask_2026 = (final_statistics_oevk["year"] == 2026)
        transformed = transform_media(
            final_statistics_oevk.loc[mask_2026, col],
            final_statistics_oevk.loc[mask_2026, "pop_total"],
            all_w3_transform_rule[metric]
        )
        log_val = float(transformed.mean())

        all_w3_rows.append(
            {
                "outlet": outlet,
                "feature": nice_name,
                "log_value": log_val,
            }
        )

# Add pro_gov_avg for reference
pg_2026 = final_statistics_oevk[final_statistics_oevk["year"] == 2026]["pro_gov_avg"]
if pg_2026.notna().any():
    all_w3_rows.append({
        "outlet": "pro_gov_avg",
        "feature": "Pro-gov Average (REF)",
        "log_value": float(pg_2026.mean())
    })

all_w3_ranked = pd.DataFrame(all_w3_rows).sort_values("log_value", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8))
plot_all_w3 = all_w3_ranked.sort_values("log_value", ascending=True)

def get_color(row):
    if "REF" in row["feature"]:
        return "#d1495b"  # red
    if any(pg in row["outlet"].lower() for pg in pro_gov_outlets):
        return "#f28e2b"  # orange
    return "#c8c8c8"  # grey

colors = [get_color(row) for _, row in plot_all_w3.iterrows()]

ax.barh(plot_all_w3["feature"], plot_all_w3["log_value"], color=colors)
ax.set_title("", fontsize=14)
ax.set_xlabel("Log-transformed Usage (average of OEVKs)", fontsize=12)
ax.grid(axis="x", alpha=0.3)

# simple color legend
legend_handles = [
    Patch(facecolor="#f28e2b", label="Pro-government outlet"),
    Patch(facecolor="#d1495b", label="Pro-government average"),
    Patch(facecolor="#c8c8c8", label="Other outlet"),
]
ax.legend(handles=legend_handles, frameon=False, fontsize=11, loc="lower right")

plt.tight_layout()

# save figure to both requested folders
if "project_root" not in locals():
    try:
        here = Path(__file__).resolve().parent
    except NameError:
        here = Path.cwd()

    project_root = here
    while not (project_root / "data").exists() and project_root != project_root.parent:
        project_root = project_root.parent

output_fig_paths = [
    project_root / "graphs_tables" / "media_outlet_w3_usage_2026.png",
    project_root / "doc" / "ÚJLaTex_ENGLISH_Template másolat 2" / "figures" / "media_outlet_w3_usage_2026.png",
]
for output_fig_path in output_fig_paths:
    output_fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_fig_path, dpi=300, bbox_inches="tight")

plt.show()

print("Saved figure to:")
for output_fig_path in output_fig_paths:
    print(output_fig_path)

# Clean up
for var in ["all_w3_rows", "pg_2026", "colors", "pro_gov_outlets", "all_w3_ranked", "plot_all_w3", "fig", "ax", "legend_handles", "output_fig_paths", "output_fig_path", "here"]:
    if var in locals():
        del locals()[var]
del var


Interpretation. Very high correlations suggest the variables carry overlapping information.
